# SQL　テーブル設計の最適化

## 目的
- テーブル正規化　(Dimension / Fact テーブル設計)　を学ぶ
- インデックスでクエリを高速化する
- パフォーマンスを実際に計測して比較する
- 複数テーブルのJOIN を実践する

## なぜテーブル設計が重要か
Day 3-4では１つのテーブルに全データを入れた。
「時間情報」「価格情報」「集計情報」を別テーブルに分けることで、
データの重複を防ぎ、メンテナンス性と検索速度を向上させる。
これを「正規化」という。


In [1]:
# ライブラリのインポート
import sqlite3
import pandas as pd
import time

conn = sqlite3.connect('nikkei225.db')

# 既存データ確認
count = pd.read_sql("SELECT COUNT(*) as cnt FROM raw_nikkei225", conn)
print(f"✓ データベース接続完了: {count['cnt']} 行のデータ")

✓ データベース接続完了: 0    1766
Name: cnt, dtype: int64 行のデータ


## 1. テーブル正規化 : Dimension テーブルとFact テーブル

###　正規化とは
データの重複をなくすためにテーブルを分割すること

### 設計パターン: Star Schema (スタースキーマ)
- Dimension テーブル: 「いつ」「何」などの属性情報（変わりにくい）
- Fact テーブル:「いくら」「何回」などの計測値（日々増える）

### 今回の設計
- dim date:日付の属性(年、月、曜日、四半期など)
- fact_nikkei225:価格データ（始値、終値、出来高など）

### なぜ分けるのか
例えば「2024年の金曜日だけの平均株価」を知りたいとき、
日付テーブルに曜日情報があれば一発で検索できる。
raw テーブルだと毎回日付を解析する計算が必要になる。


In [3]:
# Dimension テーブル:日付情報
conn.execute("DROP TABLE IF EXISTS dim_date")

conn.execute("""
             CREATE TABLE dim_date (
                date TEXT PRIMARY KEY,
                year INTEGER,
                month INTEGER,
                day INTEGER,
                quarter INTEGER,
                day_of_week INTEGER,
                day_name TEXT,
                is_month_start INTEGER,
                is_month_end INTEGER,
                is_quarter_start INTEGER
             )
""")

# raw_nikkei225 から日付情報を生成して INSERT
conn.execute("""
             INSERT INTO dim_date
             SELECT
                date,
                CAST(SUBSTR(date, 1, 4) AS INTEGER) as year,
                CAST(SUBSTR(date, 6, 2) AS INTEGER) as month,
                CAST(SUBSTR(date, 9, 2) AS INTEGER) as day,
                CASE 
                    WHEN CAST(SUBSTR(date, 6, 2) AS INTEGER) <= 3 THEN 1
                    WHEN CAST(SUBSTR(date, 6, 2) AS INTEGER) <= 6 THEN 2
                    WHEN CAST(SUBSTR(date, 6, 2) AS INTEGER) <= 9 THEN 3
                    ELSE 4
            END as quarter,
            CAST(strftime('%w', date) AS INTEGER) as day_of_week,
            CASE CAST(strftime('%w', date) AS INTEGER)
                WHEN 0 THEN 'Sunday'
                WHEN 1 THEN 'Monday'
                WHEN 2 THEN 'Tuesday'
                WHEN 3 THEN 'Wednesday'
                WHEN 4 THEN 'Thursday'
                WHEN 5 THEN 'Friday'
                WHEN 6 THEN 'Saturday'
            END as day_name,
            CASE WHEN SUBSTR(date, 9, 2) = '01' THEN 1 ELSE 0 END as is_month_start,
            0 as is_month_end,
            CASE WHEN SUBSTR(date, 9, 2) = '01' 
                 AND CAST(SUBSTR(date, 6, 2) AS INTEGER) IN (1, 4, 7, 10) 
                 THEN 1 ELSE 0 END as is_quarter_start
    FROM raw_nikkei225
""")
conn.commit()

# 確認
dim_sample = pd.read_sql("SELECT * FROM dim_date LIMIT 5", conn)
print("【dim_date テーブル（最初の5行）】")
print(dim_sample.to_string(index=False))

dim_count = pd.read_sql("SELECT COUNT(*) as cnt FROM dim_date", conn)
print(f"\n✓ dim_date テーブル作成完了: {dim_count['cnt'][0]} 行")

【dim_date テーブル（最初の5行）】
      date  year  month  day  quarter  day_of_week  day_name  is_month_start  is_month_end  is_quarter_start
2019-01-04  2019      1    4        1            5    Friday               0             0                 0
2019-01-07  2019      1    7        1            1    Monday               0             0                 0
2019-01-08  2019      1    8        1            2   Tuesday               0             0                 0
2019-01-09  2019      1    9        1            3 Wednesday               0             0                 0
2019-01-10  2019      1   10        1            4  Thursday               0             0                 0

✓ dim_date テーブル作成完了: 1766 行


## 2. Fact テーブルの作成
価格データをDimension テーブルと紐づけられる形に整理する。

In [4]:
# Fact テーブル:価格データ
conn.execute("DROP TABLE IF EXISTS fact_nikkei225")

conn.execute("""
             CREATE TABLE fact_nikkei225 (
             date TEXT PRIMARY KEY,
             open REAL,
             high REAL,
             low REAL,
             close REAL,
             volume INTEGER,
             daily_return REAL,
             FOREIGN KEY (date) REFERENCES dim_date(date)
            )
""")

# daily_returnも一緒に計算して INSERT
conn.execute("""
             INSERT INTO fact_nikkei225
             SELECT
                date,
                Open as open,
                High as high,
                Low as low,
                Close as close,
                Volume as volume,
                ROUND(
                    (Close - LAG(Close) OVER (ORDER BY date))
                / LAG(Close) OVER (ORDER BY date) * 100,
                4
             ) as daily_return
             FROM raw_nikkei225
             ORDER BY date
""")
conn.commit()

fact_sample = pd.read_sql("SELECT * FROM fact_nikkei225 ORDER BY date DESC LIMIT 5", conn)
print("【fact_nikkei225 テーブル（最新5行）】")
print(fact_sample.to_string(index=False))

fact_count = pd.read_sql("SELECT COUNT(*) as cnt FROM fact_nikkei225", conn)
print(f"\n✓ fact_nikkei225 テーブル作成完了: {fact_count['cnt'][0]} 行")

【fact_nikkei225 テーブル（最新5行）】
      date         open         high          low        close    volume  daily_return
2026-04-02 54066.828125 54258.480469 52415.109375 52469.820312         0       -2.3630
2026-04-01 51959.468750 53739.679688 51902.839844 53739.679688 165900000        5.2404
2026-03-31 51382.531250 52169.011719 50558.910156 51063.718750 174500000       -1.5845
2026-03-30 52054.679688 52054.679688 50566.988281 51885.851562 183100000       -2.7865
2026-03-27 53239.589844 53714.898438 52516.921875 53373.070312 170500000       -0.4302

✓ fact_nikkei225 テーブル作成完了: 1766 行


## 3. JOIN : 複数テーブルを結合してクエリ

正規化でテーブルをわけたら、JOINで結合して使う。
「dim_dateの曜日情報」と「fact_nikkei225の価格データ」を
合わせて分析できる。

In [6]:
# JOIN 1 : 曜日別の平均リターン
print("【JOIN クエリ　1】曜日別の平均日次リターン")
q1 = pd.read_sql("""
                SELECT
                    d.day_name,
                    d.day_of_week,
                    COUNT(*) as trading_days,
                    ROUND(AVG(f.daily_return), 4) as avg_return,
                    ROUND(AVG(f.close), 2) as avg_close
                  FROM fact_nikkei225 f
                  JOIN dim_date d ON f.date = d.date
                  WHERE f.daily_return IS NOT NULL
                  GROUP BY d.day_name, d.day_of_week
                  ORDER BY d.day_of_week
""", conn)
print(q1.to_string(index=False))

# JOIN 2:四半期別のパフォーマンス
print("\n 【JOIN クエリ 2】四半期別の平均リターン (年×四半期) ")
q2 = pd.read_sql("""
                SELECT
                  d.year,
                  d.quarter,
                  COUNT(*) as days,
                  ROUND(AVG(f.daily_return), 4) as avg_return,
                  ROUND(MAX(f.close), 2) as max_close,
                  ROUND(MIN(f.close), 2) as min_close
                FROM fact_nikkei225 f
                JOIN dim_date d ON f.date = d.date
                WHERE f.daily_return IS NOT NULL
                GROUP BY d.year, d.quarter
                ORDER BY d.year DESC, d.quarter DESC
                LIMIT 8
""", conn)
print(q2.to_string(index=False))

# ===== JOIN 3：金曜日だけの分析 =====
print("\n【JOIN クエリ 3】金曜日の年別平均リターン")
q3 = pd.read_sql("""
    SELECT 
        d.year,
        COUNT(*) as fridays,
        ROUND(AVG(f.daily_return), 4) as avg_friday_return
    FROM fact_nikkei225 f
    JOIN dim_date d ON f.date = d.date
    WHERE d.day_name = 'Friday'
      AND f.daily_return IS NOT NULL
    GROUP BY d.year
    ORDER BY d.year
""", conn)
print(q3.to_string(index=False))

【JOIN クエリ　1】曜日別の平均日次リターン
 day_name  day_of_week  trading_days  avg_return  avg_close
   Monday            1           325     -0.0057   30836.73
  Tuesday            2           359      0.2635   31149.14
Wednesday            3           363      0.0356   31049.00
 Thursday            4           359      0.0339   31186.18
   Friday            5           359     -0.0078   31009.48

 【JOIN クエリ 2】四半期別の平均リターン (年×四半期) 
 year  quarter  days  avg_return  max_close  min_close
 2026        2     2      1.4387   53739.68   52469.82
 2026        1    58      0.0426   58850.27   51063.72
 2025        4    62      0.1950   52411.34   44550.85
 2025        3    62      0.1729   45754.93   39459.62
 2025        2    62      0.2282   40487.39   31136.58
 2025        1    57     -0.1921   40083.30   35617.56
 2024        4    63      0.0862   40281.16   37808.76
 2024        3    62     -0.0318   42224.02   31458.42

【JOIN クエリ 3】金曜日の年別平均リターン
 year  fridays  avg_friday_return
 2019       50           

## 4. インデックスとはパフォーマンス計測

### インデックスとは
本の牽引と同じ。データベースが特定の値を素早く見つけるための「目次」。
インデックスなし→全行を順番に調べる (フルスキャン)
インデックスあり→目次から直接ジャンプ(高速検索)

In [8]:
#パフォーマンス計測

#テスト用:大量クエリを繰り返して時間を計測
def measure_query_time(query, conn, iterations=100):
    """クエリをN回実行して平均時間を計測"""
    start = time.time()
    for _ in range(iterations):
        pd.read_sql(query, conn)
    elapsed = (time.time() - start) / iterations * 1000 # ミリ秒
    return elapsed
# インセックスなしの計測
conn.execute("DROP INDEX IF EXISTS idx_fact_date")
conn.execute("DROP INDEX IF EXISTS idx_dim_year")

query_test = """
    SELECT d.year, AVG(f.close) as avg_close
    FROM fact_nikkei225 f
    JOIN dim_date d ON f.date = d.date
    WHERE d.year = 2024
    GROUP BY d.year
"""

time_without = measure_query_time(query_test, conn)
print(f" 【インデックスなし】{time_without:.2f} ms / クエリ")

#インデックスを作成
conn.execute("CREATE INDEX idx_fact_date on fact_nikkei225(date)")
conn.execute("CREATE INDEX idx_dim_year ON dim_date(year)")
conn.commit()

time_with = measure_query_time(query_test, conn)
print(f"【インデックスあり】{time_with:.2f} ms / クエリ")

improvement = (time_without - time_with) / time_without * 100
print(f"\n  改善率: {improvement:.1f}%")
if time_with < time_without:
    print(f"  → インデックスにより {improvement:.1f}% 高速化")
else:
    print(f"  → データ量が少ないため差が出にくい（本番では数百万行で大きな差が出る）")

 【インデックスなし】0.58 ms / クエリ
【インデックスあり】0.46 ms / クエリ

  改善率: 19.4%
  → インデックスにより 19.4% 高速化


## 5.正規化の全体像を確認

In [9]:
# 全テーブルの一覧と行数
tables = pd.read_sql("""
                    SELECT name FROM sqlite_master
                    WHERE type = 'table'
                    ORDER BY name
""", conn)

print("【データベース内の全テーブル】\n")
for table_name in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM [{table_name}]", conn)
    columns = pd.read_sql(f"PRAGMA table_info([{table_name}])", conn)
    col_names = ', '.join(columns['name'].tolist())
    print(f"  {table_name}")
    print(f"    行数: {count['cnt'][0]}")
    print(f"    列:   {col_names}\n")

【データベース内の全テーブル】

  dim_date
    行数: 1766
    列:   date, year, month, day, quarter, day_of_week, day_name, is_month_start, is_month_end, is_quarter_start

  fact_nikkei225
    行数: 1766
    列:   date, open, high, low, close, volume, daily_return

  processed_monthly
    行数: 88
    列:   month, avg_close, max_close, min_close, trading_days, price_range_pct

  raw_nikkei225
    行数: 1766
    列:   date, Open, High, Low, Close, Volume

  with_moving_avg
    行数: 1766
    列:   date, Close, ma_20, ma_200



# 6. まとめ

###今日学んだこと

正規化(Dimension /Factテーブル)　とは、データの重複をなくすためにテーブルを分割する設計手法
dim_date(日付の属性:曜日、四半期など)と fact_nikkei225(価格データ)を分けることで、
「金曜日だけ」「Q4だけ」といった絞り込みが簡単になる。
1つのテーブルに全部入れると毎回日付を解析する必要があるが、
正規化しておけば WHERE day_name = 'Friday'と書くだけで済む。

JOINとは、分けたテーブルを共通の列（今回はdate）でよこに結合する操作。
正規化でテーブルを分ける→JOINで必要な時だけ結合する、がセットの考え方。

インデックスとは、データベースの「目次」。特定の列に対して作成すると、
その列での検索が高速になる。本の索引と同じで、全ページめくる代わりに
索引から直接該当ページにジャンプできるイメージ。

なぜテーブルを分けるのか：メンテナンス性（日付属性を変えても価格データに影響しない）、
検索速度（インデックスと組み合わせて高速化）、拡張性（新しい属性を追加しやすい）のため。

###　出力結果について

曜日別リターン:火曜日のリターンが +0.2635% と他の曜日より明らかに高い。
仮説としては、月曜日に売られすぎた（-0.0057%）分を
指標を見た市場参加者が火曜日に買い戻している可能性がある。
ただし、これはあくまで仮説であり、統計検定（曜日間のリターン差のt検定）で
有意性を確認しない限り断定はできない。

四半期別パフォーマンス:2026年Q1は最高値58,850円から51,063円まで
約13%の下落幅があり、ボラティリティが大きい。
地政学リスク（中東情勢、関税政策など）の影響が考えられる。

金曜日の年別リターン:年によってプラスとマイナスが入れ替わり、
「金曜日は上がる/下がる」という一貫したパターンは見られない。

インデックスの効果: 約20%の高速化を確認（0.58ms → 0.46ms）。
今回は2,000件弱のデータなので絶対値としては小さいが、
実務で数百万〜数千万行のデータを扱う場合、この差は大きくなる。

In [10]:
conn.close()
print("✓ データベース接続を閉じました")
print("✓ Notebook 6 完了")

✓ データベース接続を閉じました
✓ Notebook 6 完了
